# Markov Decision Process in the World 8x8

## Construct states, actions and rewards
### List of states: 8x8

This code creates a list of all possible states in an 8x8 grid world. Each state is represented as a coordinate tuple (x, y), where x is the row and y is the column. This is a common way to define the state space in grid-based Markov Decision Processes (MDPs) used in Reinforcement Learning.

In [1]:
states = []
for x in range(8):                            #Iterates over all possible x-coordinates (rows) from 0 to 7
    for y in range(8):                        ##Iterates over all possible y-coordinates (columns) from 0 to 7
        states.append((x, y))
print(type(states))
print(states)

<class 'list'>
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (7, 0), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (7, 6), (7, 7)]


### Dict of actions
This code defines the action space for an 8x8 grid-world environment. It includes 8 possible moves: the four cardinal directions (Up, Down, Left, Right) and the four diagonal directions. Each action is mapped to a delta (change in position) as a tuple (dx, dy).

R = Right,
L = Left,
U = Up,
D = Down,
UR = Up-Right (Diagonal),
UL = Up-Left (Diagonal),
DR = Down-Right (Diagonal),
DL = Down-Left (Diagonal).

This allows the agent to move in 8 possible directions (like a king in chess).

In [2]:
# Define the action space as a dictionary
# Key   = Action name (abbreviation)
# Value = (dx, dy) movement delta on the grid
actions = {
    "R":  (0,  1),   # R  = Right     → move right (increase column)
    "L":  (0, -1),   # L  = Left      → move left  (decrease column)
    "U":  (-1, 0),   # U  = Up        → move up    (decrease row)
    "D":  (1,  0),   # D  = Down      → move down  (increase row)
    "UR": (-1, 1),   # UR = Up-Right   → diagonal move
    "UL": (-1,-1),   # UL = Up-Left    → diagonal move
    "DR": (1,  1),   # DR = Down-Right → diagonal move
    "DL": (1, -1)    # DL = Down-Left  → diagonal move
}

# Print information about the actions dictionary
print(type(actions))   # Shows that 'actions' is a dictionary
print(actions)         # Displays all actions and their movement deltas

<class 'dict'>
{'R': (0, 1), 'L': (0, -1), 'U': (-1, 0), 'D': (1, 0), 'UR': (-1, 1), 'UL': (-1, -1), 'DR': (1, 1), 'DL': (1, -1)}


### Rewards for usual fields (-1), bad fields (-3) and the target (10)

This reward scheme follows standard practices in Reinforcement Learning for navigation tasks:

**Default reward = -1:**
This is a standard "step cost" or "time penalty" used in most grid-world reinforcement learning environments. It encourages the agent to reach the goal in the minimum number of steps. Without this negative reward, the agent would have no motivation to move efficiently and might take unnecessarily long paths.

**Bad states reward = -3:**
Bad states represent obstacles, traps, or dangerous areas. Giving them a stronger penalty (-3) than normal steps (-1) helps the agent learn to avoid these positions more effectively. The extra negative value creates a clear distinction so the learning algorithm quickly associates these states with high risk.

**Target state reward = +10:**
The goal must be strongly attractive. A positive reward of +10 is large enough to outweigh the accumulated negative rewards from steps and bad states. This value provides a good balance — it is high enough to drive goal-seeking behavior but not so extreme that it makes the learning unstable.



In [3]:
rewards = {}  # Initialize empty dictionary for state rewards

# Set default reward for all cells in 8x8 grid
for i in range(0, 8):           # Loop through rows 0 to 7
    for j in range(0, 8):       # Loop through columns 0 to 7
        key = str(i) + ", " + str(j)   # Create key like "0, 0", "3, 5"
        rewards[key] = -1              # Default reward for normal states

# Define and penalize bad/dangerous states
bad_states = [
    (0,1), (0,5), (1,1), (1,3), (2,3),
    (3,0), (3,4), (4,2), (4,4), (5,0),
    (5,2), (6,3), (6,6), (7,1)
]

# Apply stronger negative reward to bad states
for (i, j) in bad_states:
    key = str(i) + ", " + str(j)
    rewards[key] = -3

# Set positive reward for the target/goal state
target_state = (7, 7)
key = str(target_state[0]) + ", " + str(target_state[1])
rewards[key] = 10

# Re-verify the dictionary format
print(list(rewards.keys())[:10])  # Print first 10 keys to check

['0, 0', '0, 1', '0, 2', '0, 3', '0, 4', '0, 5', '0, 6', '0, 7', '1, 0', '1, 1']


#Transitions

This dictionary represents the state transition model in the Markov Decision Process (MDP).

**Key:** State string in format "row, column" (e.g., "0, 0", "7, 7")
Value: Another dictionary that will store action → next_state mappings

**Why this structure is used:**

It allows easy lookup: transitions["3, 4"]["R"] will later give the next state after moving Right from position (3,4).
Using string keys (same format as the rewards dictionary) keeps everything consistent and easy to work with.
Dictionary comprehension is a clean, Pythonic way to initialize all 64 states at once.
Empty inner dicts {} are placeholders — you will populate them later based on the actions (R, L, U, D, etc.) and boundary checking.

In [4]:
transitions = {"{}, {}".format(*state): {} for state in states}       #Create the transitions dictionary using dictionary comprehension
print(type(transitions))
print(transitions)

<class 'dict'>
{'0, 0': {}, '0, 1': {}, '0, 2': {}, '0, 3': {}, '0, 4': {}, '0, 5': {}, '0, 6': {}, '0, 7': {}, '1, 0': {}, '1, 1': {}, '1, 2': {}, '1, 3': {}, '1, 4': {}, '1, 5': {}, '1, 6': {}, '1, 7': {}, '2, 0': {}, '2, 1': {}, '2, 2': {}, '2, 3': {}, '2, 4': {}, '2, 5': {}, '2, 6': {}, '2, 7': {}, '3, 0': {}, '3, 1': {}, '3, 2': {}, '3, 3': {}, '3, 4': {}, '3, 5': {}, '3, 6': {}, '3, 7': {}, '4, 0': {}, '4, 1': {}, '4, 2': {}, '4, 3': {}, '4, 4': {}, '4, 5': {}, '4, 6': {}, '4, 7': {}, '5, 0': {}, '5, 1': {}, '5, 2': {}, '5, 3': {}, '5, 4': {}, '5, 5': {}, '5, 6': {}, '5, 7': {}, '6, 0': {}, '6, 1': {}, '6, 2': {}, '6, 3': {}, '6, 4': {}, '6, 5': {}, '6, 6': {}, '6, 7': {}, '7, 0': {}, '7, 1': {}, '7, 2': {}, '7, 3': {}, '7, 4': {}, '7, 5': {}, '7, 6': {}, '7, 7': {}}


In [5]:
def build_actions(state, prob):
    return[prob, "({}, {})".format(*state)]
a = build_actions(states[10], 0.1)
print('a: ', a)

a:  [0.1, '(1, 2)']


 ## Example: transitions for specific state

 **How the algorithm builds transitions?**

 **(1) Compute intended next state**     
u_sum = [sum(x) for x in zip(state, actions[action])]

result = tuple(u_sum)               (state + action delta)
   
 **(2)State = states[9] = (1, 3) + any action**
      
   Let state = (1, 3), then (using all 8 actions):

**R (Right):**  (1, 3) + (0, 1)   = (1, 4)

**L (Left):**   (1, 3) + (0, -1)  = (1, 2)

**U (Up):**    (1, 3) + (-1, 0)  = (0, 3)

**D (Down):**   (1, 3) + (1, 0)   = (2, 3)

**UR (Up-Right)**:   (1, 3) + (-1, 1)  = (0, 4)

**UL (Up-Left):**   (1, 3) + (-1,-1)  = (0, 2)

**DR (Down-Right):** (1, 3) + (1, 1)   = (2, 4)

**DL (Down-Left):**  (1, 3) + (1,-1)   = (2, 2)


**(3) What is T[0][0] before before and update?**
   
   Example transition list T before update (for action "U"):
[[0.0, (0, 3)], [0.1, (1, 3)], [0.1, (1, 4)], [0.1, (1, 2)], ...] (one entry for each possible slip)

T[0][0] = 0.0 (placeholder for intended action)
T[0][1] = (0, 3) (intended next state)

If count = 0.7 (7 slips × 0.1), after update:
T[0][0] = 1.0 - 0.7 = 0.3  
   
   
   **(4) Function build_actions**
   
  build_actions(result, 0.0) → [0.0, (0, 3)]

build_actions(state, 0.1)  → [0.1, (1, 3)]
    
   

In [6]:
'''
 format of transitions:
 (x,y) = {"R: [prob, (x1, y1)]"}
'''

# Select a test state (states[9] = (1, 3))
state = states[9]   # (1, 3)

# Loop over every available action
for action in actions:
    count = 0.0                    # Reset slip probability accumulator

    # Compute intended next position: state + action delta
    result = tuple((sum(x) for x in zip(state, actions[action])))

    T = []                         # Initialize list for [prob, next_state] entries

    # Check if intended move is valid (inside grid)
    if result in states:
        T.append(build_actions(result, 0.0))   # Placeholder for main action
        T.append(build_actions(state, 0.1))    # 10% chance to stay in place
        count += 0.1
    else:
        T.append(build_actions(state, 0.0))    # Invalid move → stay put

    # Add 10% slip probability for all other actions
    for a in actions:
        if a != action:                        # Skip the intended action
            result2 = tuple((sum(x) for x in zip(state, actions[a])))
            if result2 in states:
                count += 0.1
                T.append(build_actions(result2, 0.1))

    # Assign remaining probability to the intended action
    T[0][0] = 1.0 - count

    # Store the transition list in the transitions dictionary
    transitions["{}, {}".format(*state)][action] = T

    # Print for debugging / verification
    print('state: ', state, ', action: ', action, ', transitions: ', T)


state:  (1, 1) , action:  R , transitions:  [[0.20000000000000007, '(1, 2)'], [0.1, '(1, 1)'], [0.1, '(1, 0)'], [0.1, '(0, 1)'], [0.1, '(2, 1)'], [0.1, '(0, 2)'], [0.1, '(0, 0)'], [0.1, '(2, 2)'], [0.1, '(2, 0)']]
state:  (1, 1) , action:  L , transitions:  [[0.20000000000000007, '(1, 0)'], [0.1, '(1, 1)'], [0.1, '(1, 2)'], [0.1, '(0, 1)'], [0.1, '(2, 1)'], [0.1, '(0, 2)'], [0.1, '(0, 0)'], [0.1, '(2, 2)'], [0.1, '(2, 0)']]
state:  (1, 1) , action:  U , transitions:  [[0.20000000000000007, '(0, 1)'], [0.1, '(1, 1)'], [0.1, '(1, 2)'], [0.1, '(1, 0)'], [0.1, '(2, 1)'], [0.1, '(0, 2)'], [0.1, '(0, 0)'], [0.1, '(2, 2)'], [0.1, '(2, 0)']]
state:  (1, 1) , action:  D , transitions:  [[0.20000000000000007, '(2, 1)'], [0.1, '(1, 1)'], [0.1, '(1, 2)'], [0.1, '(1, 0)'], [0.1, '(0, 1)'], [0.1, '(0, 2)'], [0.1, '(0, 0)'], [0.1, '(2, 2)'], [0.1, '(2, 0)']]
state:  (1, 1) , action:  UR , transitions:  [[0.20000000000000007, '(0, 2)'], [0.1, '(1, 1)'], [0.1, '(1, 2)'], [0.1, '(1, 0)'], [0.1, '(0, 1)'

In [7]:
'''
 format of transitions:
 (x,y) = {"R: [prob, (x1, y1)]"}
'''
# Populate transitions for every state and every action
for state in states:
    for action in actions:
        count = 0.0                    # Reset slip probability accumulator

        # Compute intended next position: state + action movement delta
        result = tuple((sum(x) for x in zip(state, actions[action])))

        T = []                         # List to hold all [prob, next_state] possibilities

        # Handle intended move
        if result in states:           # Valid move inside grid boundaries
            T.append(build_actions(result, 0.0))   # Placeholder for main action probability
            T.append(build_actions(state, 0.1))    # 10% chance to stay (slip)
            count += 0.1
        else:
            T.append(build_actions(state, 0.0))    # Invalid move (out of grid) → stay in place

        # Add 10% slip probability to all other actions
        for a in actions:
            if a != action:                        # Skip the intended action
                result2 = tuple((sum(x) for x in zip(state, actions[a])))
                if result2 in states:
                    count += 0.1
                    T.append(build_actions(result2, 0.1))

        # Assign remaining probability to the intended action
        T[0][0] = 1.0 - count

        # Store the complete transition list
        transitions["{}, {}".format(*state)][action] = T

# Special terminal state handling for the goal
transitions["7, 7"] = {"EXIT": [[1.0, (7, 7)]]}   # Terminal: stay forever with probability 1.0


state:  (0, 0) , action:  R , transitions:  [[0.7, '(0, 1)'], [0.1, '(0, 0)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  L , transitions:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  U , transitions:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  D , transitions:  [[0.7, '(1, 0)'], [0.1, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  UR , transitions:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  UL , transitions:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 0) , action:  DR , transitions:  [[0.7, '(1, 1)'], [0.1, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)']]
state:  (0, 0) , action:  DL , transitions:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  (0, 1) , action:  R , transitions:  [[0.5, '(0, 2)'], [0.1, '(0, 1)'], [0.1, '(0, 0)

#Environment

**mdp = Markov Decision Process**

In [8]:
class mdp:
    def __init__(self, transitions, rewards):
        # Store the core components of the MDP
        self.transitions = transitions   # Dictionary containing state -> action -> transitions
        self.rewards = rewards           # Dictionary containing state rewards
        self.states = transitions.keys() # List of all states (derived from transitions keys)

    def get_rewards(self, state):
        """Return the reward for a given state"""
        return self.rewards[state]

    def get_actions(self, state):
        """Return the list of possible actions available in the given state"""
        return self.transitions[state].keys()

    def get_transitions(self, state, action):
        """Return the list of [prob, next_state] for a state-action pair"""
        return self.transitions[state][action]


# Create the environment instance using previously built transitions and rewards
environment = mdp(transitions, rewards)

In [9]:
print(list(rewards.keys())[:10])

['0, 0', '0, 1', '0, 2', '0, 3', '0, 4', '0, 5', '0, 6', '0, 7', '1, 0', '1, 1']


### getting rewards, actions, transitions for 4 different states

In [10]:
# Test specific states to verify rewards, available actions, and transitions
for s in ('0, 0', '1, 3', '1, 4', '7, 7'):
    print('state: ', s)

    # Get reward for current state
    rewards = environment.get_rewards(s)
    print('state: ', s, ', rewards: ', rewards)

    # Get available actions for current state
    actions = environment.get_actions(s)

    # Print transitions for each action
    for a in actions:
        trans = environment.get_transitions(s, a)
        print('state: ', s, ', action: ', a, ', transition: ', trans)

state:  0, 0
state:  0, 0 , rewards:  -1
state:  0, 0 , action:  R , transition:  [[0.7, '(0, 1)'], [0.1, '(0, 0)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  L , transition:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  U , transition:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  D , transition:  [[0.7, '(1, 0)'], [0.1, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  UR , transition:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  UL , transition:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  0, 0 , action:  DR , transition:  [[0.7, '(1, 1)'], [0.1, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)']]
state:  0, 0 , action:  DL , transition:  [[0.7, '(0, 0)'], [0.1, '(0, 1)'], [0.1, '(1, 0)'], [0.1, '(1, 1)']]
state:  1, 3
state:  1, 3 , rewards:  -3
state:  1, 3 , action:  R , transi

### get all states from environment

**For each state, actions = dict_keys(['D', 'L', 'R', 'U', 'UR', 'UL', 'DR', 'DL'])**

In [11]:
states = environment.states
print('states: ', states)

# For each state, actions = dict_keys(['D', 'L', 'R', 'U', 'UR', 'UL', 'DR', 'DL'])
actions = environment.get_actions

V1 = {s: 0 for s in states}

for k in V1.keys():
    print('state: ', k, ', value: ', V1.get(k))


states:  dict_keys(['0, 0', '0, 1', '0, 2', '0, 3', '0, 4', '0, 5', '0, 6', '0, 7', '1, 0', '1, 1', '1, 2', '1, 3', '1, 4', '1, 5', '1, 6', '1, 7', '2, 0', '2, 1', '2, 2', '2, 3', '2, 4', '2, 5', '2, 6', '2, 7', '3, 0', '3, 1', '3, 2', '3, 3', '3, 4', '3, 5', '3, 6', '3, 7', '4, 0', '4, 1', '4, 2', '4, 3', '4, 4', '4, 5', '4, 6', '4, 7', '5, 0', '5, 1', '5, 2', '5, 3', '5, 4', '5, 5', '5, 6', '5, 7', '6, 0', '6, 1', '6, 2', '6, 3', '6, 4', '6, 5', '6, 6', '6, 7', '7, 0', '7, 1', '7, 2', '7, 3', '7, 4', '7, 5', '7, 6', '7, 7'])
state:  0, 0 , value:  0
state:  0, 1 , value:  0
state:  0, 2 , value:  0
state:  0, 3 , value:  0
state:  0, 4 , value:  0
state:  0, 5 , value:  0
state:  0, 6 , value:  0
state:  0, 7 , value:  0
state:  1, 0 , value:  0
state:  1, 1 , value:  0
state:  1, 2 , value:  0
state:  1, 3 , value:  0
state:  1, 4 , value:  0
state:  1, 5 , value:  0
state:  1, 6 , value:  0
state:  1, 7 , value:  0
state:  2, 0 , value:  0
state:  2, 1 , value:  0
state:  2, 2 , va

### 4. Calculate value function

In [26]:
def value_iteration(env):
    """Perform Value Iteration to find optimal state values."""
    states = env.states                    # All states in the environment
    get_actions = env.get_actions          # Function to get available actions
    T = env.get_transitions                # Function to get transitions
    R = env.get_rewards                    # Function to get rewards

    gamma = 0.9                            # Discount factor
    epsilon = 0.00001                      # Convergence threshold

    # Initialize value function V1 to 0 for all states
    V1 = {s: 0.0 for s in states}
    j = {s: 0 for s in states}             # Counter for number of updates per state

    iteration = 0

    while True:
        V = V1.copy()                      # Copy previous value function
        delta = 0.0                        # Track maximum change
        iteration += 1

        # Update value for each state
        for s in states:
            def safe_get_value(state_key_from_transition):
                """Convert transition state format if needed and get its value"""
                # Handle cases where next_state might be in '(x, y)' tuple string format
                if isinstance(state_key_from_transition, str) and \
                   state_key_from_transition.startswith('(') and \
                   state_key_from_transition.endswith(')'):
                    state_key_from_transition = state_key_from_transition.strip('()')
                return V.get(state_key_from_transition, 0.0)

            # Compute action values (expected value for each action)
            action_values = []
            for a in get_actions(s):
                transitions = T(s, a)
                if not transitions:
                    continue
                # Bellman expectation: sum(prob * value of next state)
                expected_value = sum(p * safe_get_value(s1) for p, s1 in transitions)
                action_values.append(expected_value)

            # Bellman optimality update: V(s) = R(s) + gamma * max_a E[V(s')]
            if action_values:
                V1[s] = R(s) + gamma * max(action_values)
            else:
                V1[s] = R(s)

            # Track maximum change for convergence check
            delta = max(delta, abs(V1[s] - V[s]))
            j[s] += 1                       # Count updates for this state

        # Check convergence
        if delta < epsilon:
            print(f"Converged after {iteration} iterations")
            # Print update counts per state
            for s in states:
                print(f'state: {s}, updates: {j[s]}')
            return V1

In [16]:
#print(value_iteration(environment))

V = value_iteration(environment)
for k in V.keys():
    print('state: ', k, ', value: ', V.get(k))

Converged after 2 iterations
state: 0, 0, updates: 2
state: 0, 1, updates: 2
state: 0, 2, updates: 2
state: 0, 3, updates: 2
state: 0, 4, updates: 2
state: 0, 5, updates: 2
state: 0, 6, updates: 2
state: 0, 7, updates: 2
state: 1, 0, updates: 2
state: 1, 1, updates: 2
state: 1, 2, updates: 2
state: 1, 3, updates: 2
state: 1, 4, updates: 2
state: 1, 5, updates: 2
state: 1, 6, updates: 2
state: 1, 7, updates: 2
state: 2, 0, updates: 2
state: 2, 1, updates: 2
state: 2, 2, updates: 2
state: 2, 3, updates: 2
state: 2, 4, updates: 2
state: 2, 5, updates: 2
state: 2, 6, updates: 2
state: 2, 7, updates: 2
state: 3, 0, updates: 2
state: 3, 1, updates: 2
state: 3, 2, updates: 2
state: 3, 3, updates: 2
state: 3, 4, updates: 2
state: 3, 5, updates: 2
state: 3, 6, updates: 2
state: 3, 7, updates: 2
state: 4, 0, updates: 2
state: 4, 1, updates: 2
state: 4, 2, updates: 2
state: 4, 3, updates: 2
state: 4, 4, updates: 2
state: 4, 5, updates: 2
state: 4, 6, updates: 2
state: 4, 7, updates: 2
state: 5, 0

### 5. Get policy

In [27]:
def policy(env):
    states= env.states
    get_actions= env.get_actions
    T = env.get_transitions
    V = value_iteration(env)


    P={}
    for s in states:
        def format_state_key_for_V(s1_from_transition):
            if isinstance(s1_from_transition, str) and s1_from_transition.startswith('(') and s1_from_transition.endswith(')'):
                return s1_from_transition.strip('()')
            return s1_from_transition

        P[s] = max(get_actions(s),  key= lambda a:sum([p*V[format_state_key_for_V(s1)]  for (p,s1) in T(s,a)]))
    return P

In [28]:
P = policy(environment)

print('type policy: ', type(P), ', type of P.keys(): ', type(P.keys()))
sorted_keys = sorted(P.keys())
for k in sorted_keys:
    print('key: ', k, ', best action: ', P[k])

Converged after 86 iterations
state: 0, 0, updates: 86
state: 0, 1, updates: 86
state: 0, 2, updates: 86
state: 0, 3, updates: 86
state: 0, 4, updates: 86
state: 0, 5, updates: 86
state: 0, 6, updates: 86
state: 0, 7, updates: 86
state: 1, 0, updates: 86
state: 1, 1, updates: 86
state: 1, 2, updates: 86
state: 1, 3, updates: 86
state: 1, 4, updates: 86
state: 1, 5, updates: 86
state: 1, 6, updates: 86
state: 1, 7, updates: 86
state: 2, 0, updates: 86
state: 2, 1, updates: 86
state: 2, 2, updates: 86
state: 2, 3, updates: 86
state: 2, 4, updates: 86
state: 2, 5, updates: 86
state: 2, 6, updates: 86
state: 2, 7, updates: 86
state: 3, 0, updates: 86
state: 3, 1, updates: 86
state: 3, 2, updates: 86
state: 3, 3, updates: 86
state: 3, 4, updates: 86
state: 3, 5, updates: 86
state: 3, 6, updates: 86
state: 3, 7, updates: 86
state: 4, 0, updates: 86
state: 4, 1, updates: 86
state: 4, 2, updates: 86
state: 4, 3, updates: 86
state: 4, 4, updates: 86
state: 4, 5, updates: 86
state: 4, 6, updates

![](policy_6x6.png)